In [3]:
!pip install torch
!pip install torchvision


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   --------------- ------------------------ 1.6/4.0 MB 4.7 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.0 MB 4.5 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 4.6 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 128
epochs = 10
lr = 1e-3
mc_samples = 20

torch.manual_seed(0)
np.random.seed(0)

In [5]:
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
])

transform_mnist = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1))
])

train_id = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_cifar)
test_id = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_cifar)

test_ood = datasets.MNIST(root="./data", train=False, download=True, transform=transform_mnist)

train_id_loader = DataLoader(train_id, batch_size=batch_size, shuffle=True)
test_id_loader = DataLoader(test_id, batch_size=batch_size, shuffle=False)
test_ood_loader = DataLoader(test_ood, batch_size=batch_size, shuffle=False)


100%|██████████| 170M/170M [01:45<00:00, 1.62MB/s]   
100%|██████████| 9.91M/9.91M [00:09<00:00, 1.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 69.8kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 525kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.20MB/s]


In [6]:
class CNN(nn.Module):
    def __init__(self, dropout_p=0.3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x, return_features=False):
        feats = self.features(x).squeeze(-1).squeeze(-1)
        feats = self.dropout(feats)
        logits = self.fc(feats)

        if return_features:
            return logits, feats
        return logits


In [7]:
def train(model, loader, epochs=10, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}: loss = {total_loss/len(loader):.4f}")


In [8]:
def compute_ood_metrics(id_scores, ood_scores):
    y_true = np.concatenate([
        np.zeros_like(id_scores),
        np.ones_like(ood_scores)
    ])
    scores = np.concatenate([id_scores, ood_scores])

    auroc = roc_auc_score(y_true, scores)
    aupr = average_precision_score(y_true, scores)

    fpr, tpr, _ = roc_curve(y_true, scores)
    fpr95 = fpr[np.where(tpr >= 0.95)[0][0]]

    return auroc, aupr, fpr95


In [9]:
def get_softmax_ood_scores(model, id_loader, ood_loader):
    model.eval()
    model.to(device)

    def collect(loader):
        scores = []
        with torch.no_grad():
            for x, _ in loader:
                x = x.to(device)
                probs = F.softmax(model(x), dim=1)
                scores.append((1 - probs.max(dim=1)[0]).cpu().numpy())
        return np.concatenate(scores)

    return collect(id_loader), collect(ood_loader)


In [10]:
def get_mcd_entropy(model, x, T=20):
    model.train()
    probs = []
    with torch.no_grad():
        for _ in range(T):
            probs.append(F.softmax(model(x), dim=1).unsqueeze(0))
    probs = torch.cat(probs, dim=0).mean(0)
    return -torch.sum(probs * torch.log(probs + 1e-8), dim=1)


def get_mcd_ood_scores(model, id_loader, ood_loader, T=20):
    model.to(device)

    def collect(loader):
        scores = []
        for x, _ in loader:
            x = x.to(device)
            scores.append(get_mcd_entropy(model, x, T).cpu().numpy())
        return np.concatenate(scores)

    return collect(id_loader), collect(ood_loader)


In [11]:
def compute_react_threshold(model, loader, percentile=90):
    model.eval()
    feats_all = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            _, feats = model(x, return_features=True)
            feats_all.append(feats.cpu().numpy())

    feats_all = np.concatenate(feats_all)
    return np.percentile(feats_all, percentile)


In [12]:
def get_react_ood_scores(model, id_loader, ood_loader, clip_value):
    model.eval()
    model.to(device)

    def collect(loader):
        scores = []
        with torch.no_grad():
            for x, _ in loader:
                x = x.to(device)
                logits, feats = model(x, return_features=True)
                feats = torch.clamp(feats, max=clip_value)
                logits = model.fc(feats)
                probs = F.softmax(logits, dim=1)
                scores.append((1 - probs.max(dim=1)[0]).cpu().numpy())
        return np.concatenate(scores)

    return collect(id_loader), collect(ood_loader)


In [13]:
model = CNN(dropout_p=0.3)
train(model, train_id_loader, epochs=epochs, lr=lr)


Epoch 1: loss = 1.8425
Epoch 2: loss = 1.5863
Epoch 3: loss = 1.4635
Epoch 4: loss = 1.3711
Epoch 5: loss = 1.3041
Epoch 6: loss = 1.2404
Epoch 7: loss = 1.1953
Epoch 8: loss = 1.1450
Epoch 9: loss = 1.1069
Epoch 10: loss = 1.0780


In [14]:
# Softmax
s_id, s_ood = get_softmax_ood_scores(model, test_id_loader, test_ood_loader)
s_metrics = compute_ood_metrics(s_id, s_ood)

# MC Dropout
m_id, m_ood = get_mcd_ood_scores(model, test_id_loader, test_ood_loader, T=mc_samples)
m_metrics = compute_ood_metrics(m_id, m_ood)

# ReAct
clip = compute_react_threshold(model, train_id_loader, percentile=90)
r_id, r_ood = get_react_ood_scores(model, test_id_loader, test_ood_loader, clip)
r_metrics = compute_ood_metrics(r_id, r_ood)

print("Softmax  AUROC / AUPR / FPR95:", s_metrics)
print("MC-Drop   AUROC / AUPR / FPR95:", m_metrics)
print("ReAct     AUROC / AUPR / FPR95:", r_metrics)

Softmax  AUROC / AUPR / FPR95: (0.69130617, 0.6040083340193275, 0.6952)
MC-Drop   AUROC / AUPR / FPR95: (0.6884751149999999, 0.5723918749544441, 0.6051)
ReAct     AUROC / AUPR / FPR95: (0.8393576300000001, 0.7881297261441829, 0.5081)


softmax и метод монте-карло слабо справляются с задачей ood-детекции при переносе с CIFAR-10 на MNIST: их значения AUROC и AUPR находятся на уровне ~0.69 и ~0.60 соответственно, а FPR95 остаётся высоким

ReAct, основанный на клиппинге активаций, демонстрирует существенно лучшее качество: заметный рост AUROC и AUPR и снижение FPR95, что указывает на более чёткое разделение ID и OOD примеров без использования байесовских предположений и без усложнения обучения